# Script 1: Basic Data Cleaning

- Rename columns for consistency and clarity  
- Generate summary statistics for article lengths to assess token window fit  
- Deduplicate articles by hashing their contents  
- Remove special LLM tokens that may affect inference or tokenization  
- Save the cleaned dataset for downstream tasks

In [ ]:
import fenic as fc
from fenic.api.session import CloudConfig, CloudExecutorSize

from dotenv import load_dotenv

import nest_asyncio
nest_asyncio.apply()

load_dotenv()

fc.configure_logging()

DATA_PATH = "s3://td-usw2-nonprod-custmock-data1/integration_test_data/test_cloud_session/medium_data.csv"
CLEAN_DATA_PATH_PARQUET = "s3://td-usw2-nonprod-custmock-data1/integration_test_data/test_cloud_session/medium_data.parquet"
TABLE_LOCATION = "s3://td-usw2-nonprod-custmock-data1/integration_test_data/test_cloud_session/sample_table.parquet"

config = fc.SessionConfig(
        app_name="medium_curation",
        db_path="test_db",
        cloud=CloudConfig(
            size=CloudExecutorSize.SMALL,
        ),
        semantic=fc.SemanticConfig(
            language_models={
                "flash": fc.GoogleGLAModelConfig(
                    model_name="gemini-2.0-flash",
                    rpm=2000,
                    tpm=4_000_000,
                ),
            },
        ),
    )

session = fc.Session.get_or_create(config)

# table_definition = session.catalog.does_table_exist("some_table")
# print(f"Table exists: {table_definition}")

data = {
        "name": ["Alice", "Bob", "Charlie", "David", None, "Alice"],
        "age": [None, 30, 30, 95, 25, 20],
        "group": [100, 300, 300, 100, 200, 300],
        "city": [
            "Product with Null",
            "San Francisco",
            "Seattle",
            "Largest Product",
            "San Francisco",
            "Denver",
        ],
    }

df = session.create_dataframe(data)
# df = session.read.parquet(CLEAN_DATA_PATH_PARQUET)
#df.limit(10).show()
row_count =df.count()

print(f"Done: {row_count}")

# Let's debug the table creation
df.write.save_as_table(table_name="some_table", mode="ignore", location=TABLE_LOCATION)

# df = session.read.csv(DATA_PATH)

# df = (
#     df.with_column_renamed("URL", "url")
#         .with_column_renamed("PUBLISHED_AT", "published_at")
#         .with_column_renamed("TAG_SLUG", "tag_slug")
#         .with_column_renamed("TITLE", "title")
#         .with_column_renamed("STORY_READS", "story_reads")
#         .with_column_renamed("TYPE_WRITER", "type_writer")
#         .with_column_renamed("TEXT", "text")
# )

# df.count()

# df.drop_duplicates(["url"]).count()

# print("Hashing and deduplicating")
# hashed = session.sql("""
#     SELECT
#         *,
#         hash(title) AS title_hash,
#         hash(text) AS text_hash,
#         CAST(CAST(published_at AS TIMESTAMP) AS DATE) AS publish_date
#     FROM {df}
# """, df=df).drop("published_at").cache()

# deduped = hashed.drop_duplicates(["title_hash", "text_hash"]).drop("title_hash", "text_hash","tag_slug").cache()
# deduped.count()

# with_article_length = (
#     deduped.with_column("article_length", fc.text.length(fc.col("text")))
#         .with_column("num_article_tokens", fc.text.count_tokens("text"))
# )

# article_body_stats = with_article_length.agg(
#     fc.min("article_length").alias("min_article_length"),
#     fc.max("article_length").alias("max_article_length"),
#     fc.avg("article_length").alias("avg_article_length"),
#     fc.stddev("article_length").alias("std_article_length"),
#     fc.min("num_article_tokens").alias("min_num_article_tokens"),
#     fc.max("num_article_tokens").alias("max_num_article_tokens"),
#     fc.avg("num_article_tokens").alias("avg_num_article_tokens"),
#     fc.stddev("num_article_tokens").alias("std_num_article_tokens"),
# )

# # Show the first 10 rows of the article body stats
# article_body_stats.limit(10).show()

# special_tokens = [
#     "<|endoftext|>",       # End of generated/completion text
#     "<|startoftext|>",     # Start of input text (sometimes used in training)
#     "<|im_start|>",        # Start of message block (chat format)
#     "<|im_end|>",          # End of message block
#     "<|system|>",          # Used in some variants to mark system instructions
#     "<|user|>",            # Marks a user message
#     "<|assistant|>",       # Marks an assistant response
#     "<|sep|>",             # General separator (used in some tasks)
#     "<|bos|>",             # Beginning of sequence (used in some tokenizers)
#     "<|eos|>",             # End of sequence (often same as <|endoftext|>)
# ]

# pattern = r"<\|.*?\|>"
# cleaned_df = deduped.with_column("text",
#         fc.text.regexp_replace(fc.col("text"), pattern, "[TOKEN]")).cache()

# cleaned_df.write.parquet("s3://td-usw2-nonprod-custmock-data1/integration_test_data/test_cloud_session/medium_data.parquet", mode="overwrite")

## Step 1: Rename columns

In [ ]:
import fenic as fc
from fenic.api.session import CloudConfig, CloudExecutorSize

from dotenv import load_dotenv

import nest_asyncio
nest_asyncio.apply()

load_dotenv()

fc.configure_logging()

# When running int he
DATA_PATH_PARQUET = "s3://td-usw2-nonprod-custmock-data1/integration_test_data/test_cloud_session/medium_data.parquet"

config = fc.SessionConfig(
        app_name="medium_curation",
        db_path="test_db",
        cloud=CloudConfig(
            size=CloudExecutorSize.SMALL,
        ),
        semantic=fc.SemanticConfig(
            language_models={
                "flash": fc.GoogleGLAModelConfig(
                    model_name="gemini-2.0-flash",
                    rpm=2000,
                    tpm=4_000_000,
                ),
            },
        ),
    )

session = fc.Session.get_or_create(config)



In [ ]:
df.drop_duplicates(["url"]).count()

## Step 2: Deduplicate articles

In [ ]:
hashed = session.sql("""
    SELECT
        *,
        hash(title) AS title_hash,
        hash(text) AS text_hash,
        CAST(CAST(published_at AS TIMESTAMP) AS DATE) AS publish_date
    FROM {df}
""", df=df).drop("published_at").cache()

deduped = hashed.drop_duplicates(["title_hash", "text_hash"]).drop("title_hash", "text_hash","tag_slug").cache()
deduped.count()

In [ ]:
with_article_length = (
    deduped.with_column("article_length", fc.text.length(fc.col("text")))
        .with_column("num_article_tokens", fc.text.count_tokens("text"))
)

article_body_stats = with_article_length.agg(
    fc.min("article_length").alias("min_article_length"),
    fc.max("article_length").alias("max_article_length"),
    fc.avg("article_length").alias("avg_article_length"),
    fc.stddev("article_length").alias("std_article_length"),
    fc.min("num_article_tokens").alias("min_num_article_tokens"),
    fc.max("num_article_tokens").alias("max_num_article_tokens"),
    fc.avg("num_article_tokens").alias("avg_num_article_tokens"),
    fc.stddev("num_article_tokens").alias("std_num_article_tokens"),
)

article_body_stats.show()

special_tokens = [
    "<|endoftext|>",       # End of generated/completion text
    "<|startoftext|>",     # Start of input text (sometimes used in training)
    "<|im_start|>",        # Start of message block (chat format)
    "<|im_end|>",          # End of message block
    "<|system|>",          # Used in some variants to mark system instructions
    "<|user|>",            # Marks a user message
    "<|assistant|>",       # Marks an assistant response
    "<|sep|>",             # General separator (used in some tasks)
    "<|bos|>",             # Beginning of sequence (used in some tokenizers)
    "<|eos|>",             # End of sequence (often same as <|endoftext|>)
]

pattern = r"<\|.*?\|>"
cleaned_df = deduped.with_column("text",
        fc.text.regexp_replace(fc.col("text"), pattern, "[TOKEN]")).cache()

cleaned_df.write.save_as_table("cleaned", mode="error")

All articles fit within the language model’s context window, but exceed the embedding model’s limits. We’ll need to summarize or extract key content from each article before generating embeddings.

## Step 3: Remove invalid tokens from articles that may affect inference/tokenization downstream

In [ ]:
special_tokens = [
    "<|endoftext|>",       # End of generated/completion text
    "<|startoftext|>",     # Start of input text (sometimes used in training)
    "<|im_start|>",        # Start of message block (chat format)
    "<|im_end|>",          # End of message block
    "<|system|>",          # Used in some variants to mark system instructions
    "<|user|>",            # Marks a user message
    "<|assistant|>",       # Marks an assistant response
    "<|sep|>",             # General separator (used in some tasks)
    "<|bos|>",             # Beginning of sequence (used in some tokenizers)
    "<|eos|>",             # End of sequence (often same as <|endoftext|>)
]

pattern = r"<\|.*?\|>"
cleaned_df = deduped.with_column("text",
        fc.text.regexp_replace(fc.col("text"), pattern, "[TOKEN]")).cache()

## Step 4: Save Results

In [ ]:
cleaned_df.write.save_as_table("cleaned", mode="error")

In [ ]:
session.stop()